In [39]:
import os
import time
import json
from typing import TypedDict, List

# --- API keys ---
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "VOTRE_CLE_GROQ_ICI")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "VOTRE_CLE_TAVILY_ICI")
GROQ_MODEL = "openai/gpt-oss-20b"


GROQ_OK = GROQ_API_KEY.startswith("gsk_") and len(GROQ_API_KEY) > 30
TAVILY_OK = TAVILY_API_KEY.startswith("tvly-") and len(TAVILY_API_KEY) > 20

print(f"Groq   : {'✅ ok' if GROQ_OK else '❌ clé manquante'}")
print(f"Tavily : {'✅ ok' if TAVILY_OK else '❌ clé manquante (web search désactivée)'}")
print(f"Modèle : {GROQ_MODEL}")

Groq   : ✅ ok
Tavily : ✅ ok
Modèle : openai/gpt-oss-20b


In [40]:
import os

# À définir AVANT d'importer transformers / sentence-transformers / huggingface_hub
os.environ["HF_HUB_VERBOSITY"] = "error"            # silence « unauthenticated requests to the HF Hub »
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # silence les barres « Loading weights »
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# --- 1. Mini AI corpus (equivalent to TP 3) ---
CORPUS = {
 "transformers.md": '''# Transformer Architecture
## Self-Attention
The Transformer architecture (Vaswani et al., 2017) replaced RNNs with self-attention.
It computes Query (Q), Key (K), Value (V) matrices and uses softmax(QK^T / sqrt(d_k)) * V.
Multi-head attention runs multiple attention heads in parallel.
## Variants
Encoder-only models (BERT), decoder-only models (GPT), encoder-decoder (T5, BART).
Modern positional encodings: RoPE (Rotary), ALiBi.''',

 "rag_systems.md": '''# Retrieval-Augmented Generation (RAG)
## Definition
RAG combines an information retrieval layer with an LLM. Responses are generated from
retrieved evidence rather than parametric memory only.
## Hybrid Retrieval
Hybrid retrieval combines sparse (BM25) and dense (embeddings) methods, fused with
Reciprocal Rank Fusion (RRF).
## Chunking
Common strategies: fixed-size, recursive, semantic, structure-aware (Markdown headers).
Optimal chunk_size depends on the use case: 256 tokens for precision, 1024 for context.''',

 "vector_databases.md": '''# Vector Databases and Indexing
## HNSW (Hierarchical Navigable Small World)
HNSW builds a multi-layer graph where each layer is a small-world network.
Search starts at the top sparse layer and refines through lower dense layers.
Achieves 95%+ recall with millisecond latency. Key params: M, ef_construction, ef_search.
## IVF (Inverted File Index)
IVF clusters vectors with k-means; search only in nearest clusters (nprobe).
Combined with PQ (product quantization) for memory savings on huge corpora.''',

 "finetuning.md": '''# LLM Fine-Tuning
## LoRA (Low-Rank Adaptation)
LoRA freezes the pre-trained model and injects trainable rank decomposition matrices
into each Transformer layer. Update is W + BA where B,A are small.
## QLoRA
QLoRA combines LoRA with 4-bit quantization. Enables fine-tuning 65B models on a single 48GB GPU.
## RLHF and DPO
RLHF: reward model + PPO. DPO: simpler, no separate reward model, classification loss on preference pairs.''',
}

from langchain_core.documents import Document
docs = [Document(page_content=text, metadata={"source": name})
        for name, text in CORPUS.items()]

# --- 2. Splitting ---
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=300, chunk_overlap=30,
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents → {len(chunks)} chunks")

# --- 3. Embeddings + Chroma ---
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Repartir propre : sans ça, ré-exécuter cette cellule empile les chunks dans la
# même collection (4 → 8 → 12) et le retriever renvoie des doublons.
try:
    Chroma(collection_name="tp4a_corpus", embedding_function=embeddings).delete_collection()
except Exception:
    pass
vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="tp4a_corpus")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Retriever prêt — collection 'tp4a_corpus' avec {vectorstore._collection.count()} chunks.")


4 documents → 4 chunks
Retriever prêt — collection 'tp4a_corpus' avec 4 chunks.


In [41]:
from langchain_core.tools import tool

@tool
def rag_search(query: str) -> str:
    """Cherche des informations dans le corpus IA local (Transformer, RAG,
    fine-tuning, vector databases). À utiliser pour les questions techniques
    sur ces sujets quand l'information est susceptible d'être dans la base."""
    docs = retriever.invoke(query)
    if not docs:
        return "[Aucun document pertinent trouvé dans le corpus local]"
    return "\n\n---\n\n".join(
        f"[source: {d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in docs
    )

# Quick test
print(rag_search.invoke("How does HNSW work?")[:400])


[source: vector_databases.md]
# Vector Databases and Indexing
## HNSW (Hierarchical Navigable Small World)
HNSW builds a multi-layer graph where each layer is a small-world network.
Search starts at the top sparse layer and refines through lower dense layers.
Achieves 95%+ recall with millisecond latency. Key params: M, ef_construction, ef_search.
## IVF (Inverted File Index)
IVF clusters vectors 


In [42]:
import arxiv
from langchain_tavily import TavilySearch
from langchain_core.tools import tool, Tool

# --- Tool 2: web search (Tavily) ---
if TAVILY_OK:
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY   # for the internal client
    # TavilySearch (LangChain 1.x) — renvoie {"results": [...]} et non une liste
    tavily_tool = TavilySearch(
        max_results=3,
        name="tavily_web_search",
        description=(
            "Recherche sur le web public. À utiliser pour les questions récentes, "
            "hors du corpus local, ou nécessitant des informations à jour."
        ),
    )
else:
    # Stub pour que le notebook tourne sans clé Tavily
    tavily_tool = Tool.from_function(
        func=lambda q: "[Tavily désactivé — pas de clé API]",
        name="tavily_web_search",
        description="Recherche web (désactivée).",
    )

# --- Tool 3: arXiv (papiers scientifiques) ---
@tool
def arxiv_search(query: str) -> str:
    """Cherche des papiers scientifiques sur arXiv. À utiliser pour les questions
    concernant des publications académiques précises (auteur, titre, papier précis)."""
    try:
        papers = list(arxiv.Client().results(arxiv.Search(query=query, max_results=3)))
    except Exception as e:                      # réseau / API indisponible
        return f"[arXiv indisponible : {e}]"
    if not papers:
        return "[Aucun papier arXiv trouvé pour cette requête]"
    return "\n\n---\n\n".join(
        f"[{p.published:%Y-%m-%d}] {p.title}\n"
        f"Auteurs : {', '.join(a.name for a in p.authors)}\n{p.summary.strip()}"
        for p in papers
    )

tools = [rag_search, tavily_tool, arxiv_search]
for t in tools:
    print(f" • {t.name:<25} — {t.description[:80]}...")


 • rag_search                — Cherche des informations dans le corpus IA local (Transformer, RAG,
    fine-tun...
 • tavily_web_search         — Recherche sur le web public. À utiliser pour les questions récentes, hors du cor...
 • arxiv_search              — Cherche des papiers scientifiques sur arXiv. À utiliser pour les questions
    c...


In [43]:
SYSTEM_PROMPT = """Tu es un AI Research Assistant spécialisé en intelligence artificielle.
Tu disposes de 3 outils :
- `rag_search` : corpus IA local (Transformer, RAG, HNSW, LoRA, vector DBs).
- `tavily_web_search` : recherche web généraliste (infos récentes 2024-2026).
- `arxiv_search` : papiers scientifiques arXiv (publications académiques précises).

RÈGLES :
- Utilise `rag_search` en priorité pour les concepts fondamentaux IA déjà bien établis
  (Transformer, RAG, HNSW, LoRA, embeddings...).
- Utilise `tavily_web_search` pour les informations récentes (2024-2026) ou hors corpus local.
- Utilise `arxiv_search` quand la question concerne un papier précis ou des publications
  académiques.
- Refuse poliment les questions hors-domaine (cuisine, météo, etc.) en français,
  SANS appeler aucun outil.
- Cite tes sources dans ta réponse finale.
- Réponds en français.
"""
print(SYSTEM_PROMPT[:300], "...")

Tu es un AI Research Assistant spécialisé en intelligence artificielle.
Tu disposes de 3 outils :
- `rag_search` : corpus IA local (Transformer, RAG, HNSW, LoRA, vector DBs).
- `tavily_web_search` : recherche web généraliste (infos récentes 2024-2026).
- `arxiv_search` : papiers scientifiques arXiv  ...


In [44]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

# reasoning_effort="low" + max_tokens : le tier gratuit plafonne à 8 000 TPM
llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY, temperature=0,
               max_tokens=1024, reasoning_effort="low")

# LangChain 1.x ReAct agent: returns a compiled LangGraph (not an AgentExecutor)
agent_executor = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)
print(f"Agent ReAct prêt avec {len(tools)} outils.")


Agent ReAct prêt avec 3 outils.


In [45]:
golden_set = [
    {"id": "Q1",
     "question": "Comment fonctionne l'algorithme HNSW pour la recherche approximative de plus proches voisins ?",
     "expected_tools": ["rag_search"]},
    {"id": "Q2",
     "question": "Quels nouveaux modèles d'embedding open-source sont sortis en 2025 ?",
     "expected_tools": ["tavily_web_search"]},
    {"id": "Q3",
     "question": "Que dit le papier original LoRA (2021) sur l'efficacité paramétrique du fine-tuning ?",
     "expected_tools": ["arxiv_search"]},
    {"id": "Q4",
     "question": "Compare HNSW à des alternatives utilisées en production en 2024-2025.",
     "expected_tools": ["rag_search", "tavily_web_search"]},
    {"id": "Q5",
     "question": "Quelle est la recette préférée de l'auteur du papier 'Attention is all you need' ?",
     "expected_tools": []},   # a polite refusal is expected
]
print(f"{len(golden_set)} questions du golden set prêtes.")

5 questions du golden set prêtes.


In [46]:
def run_react_agent(question: str) -> dict:
    """Run the ReAct agent (LangGraph prebuilt) and capture trace + metrics.

    LangGraph output: result["messages"] = ordered list
    [HumanMessage, AIMessage(tool_calls=...), ToolMessage, ..., AIMessage final].
    """
    if not GROQ_OK:
        return {"answer": "[Pas de clé Groq]", "tools_used": [], "steps": 0, "duration_s": 0}

    t0 = time.time()
    try:
        result = agent_executor.invoke(
            {"messages": [("user", question)]},
            {"recursion_limit": 12},      # anti-loop guardrail
        )
        duration = time.time() - t0
        messages = result["messages"]

        # Tools called (from AIMessages carrying tool_calls)
        tools_used = []
        for msg in messages:
            for tc in getattr(msg, "tool_calls", []) or []:
                tools_used.append(tc["name"])

        # Final answer = last non-empty AIMessage
        final_answer = ""
        for msg in reversed(messages):
            if msg.__class__.__name__ == "AIMessage" and getattr(msg, "content", ""):
                final_answer = msg.content
                break

        # Verbose ReAct-style display
        for msg in messages:
            cls = msg.__class__.__name__
            if cls == "AIMessage":
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  🤖 Action : {tc['name']}({tc['args']})")
                elif msg.content:
                    print(f"  💬 Final : {msg.content[:200]}{'…' if len(msg.content) > 200 else ''}")
            elif cls == "ToolMessage":
                preview = str(msg.content)[:160].replace("\n", " ")
                print(f"  📥 Observation [{msg.name}] : {preview}…")

        return {
            "answer": final_answer,
            "tools_used": tools_used,
            "steps": len(tools_used),
            "duration_s": round(duration, 2),
        }
    except Exception as e:
        print(f"  ⚠️ Erreur : {e}")
        return {"answer": f"[Erreur : {e}]", "tools_used": [], "steps": 0,
                "duration_s": round(time.time() - t0, 2)}

In [47]:
react_results = {}
for item in golden_set:
    print("\n" + "=" * 80)
    print(f"### {item['id']} : {item['question']}")
    print("=" * 80)
    r = run_react_agent(item["question"])
    # Tier gratuit : 8 000 TPM sur gpt-oss-20b -> pause puis un nouvel essai
    if "rate_limit" in r["answer"] or "413" in r["answer"]:
        print("  ⏳ Limite TPM atteinte — pause 60 s puis nouvel essai")
        time.sleep(60)
        r = run_react_agent(item["question"])
    react_results[item["id"]] = r
    print(f"\n📊 Outils utilisés : {r['tools_used']}")
    print(f"   Durée : {r['duration_s']}s | Étapes : {r['steps']}")



### Q1 : Comment fonctionne l'algorithme HNSW pour la recherche approximative de plus proches voisins ?
  🤖 Action : rag_search({'query': 'HNSW algorithm approximate nearest neighbor search explanation'})
  📥 Observation [rag_search] : [source: vector_databases.md] # Vector Databases and Indexing ## HNSW (Hierarchical Navigable Small World) HNSW builds a multi-layer graph where each layer is a…
  💬 Final : ### Algorithme HNSW (Hierarchical Navigable Small World)

HNSW est un index de recherche de plus proches voisins (ANN) qui combine deux idées :

| Concept | Description |
|---------|-------------|
| *…

📊 Outils utilisés : ['rag_search']
   Durée : 1.65s | Étapes : 1

### Q2 : Quels nouveaux modèles d'embedding open-source sont sortis en 2025 ?
  🤖 Action : tavily_web_search({'query': "nouveaux modèles d'embedding open-source 2025", 'search_depth': 'advanced', 'time_range': 'year'})
  📥 Observation [tavily_web_search] : {"query": "nouveaux modèles d'embedding open-source 2025", "fol

## Étape 8 — Analyse des résultats

Exécution de référence : `openai/gpt-oss-20b`, `temperature=0`, `reasoning_effort="low"`, `max_tokens=1024`, `recursion_limit=12`.
Les durées et les choix d'outils varient d'une exécution à l'autre, même à temperature 0 : ce tableau décrit **cette** exécution, pas une vérité reproductible.

| Q | Attendu | Observé | Verdict | Diagnostic |
|---|---------|---------|---------|------------|
| Q1 | `rag_search` | `rag_search`, 1 étape, 1.72 s | ✅ Exact | Bonne route, réponse fidèle à `vector_databases.md` (couches, `M`, `ef_construction`, `ef_search`). |
| Q2 | `tavily_web_search` | `tavily_web_search`, 1 étape, 2.47 s | ✅ Exact | Bon outil, sources citées. Réserve : `Llama-Embed-Nemotron-8B` est sous licence non commerciale, donc « open-source » mérite nuance. |
| Q3 | `arxiv_search` | `arxiv_search` + `rag_search`, 2 étapes, 26.82 s | ≈ Partiel | Bon premier outil mais requête trop générique → papier 2025 hors sujet ; l'agent se rattrape seul avec `rag_search` (outil non attendu) puis cite un titre erroné (« Low-Rank Adaptation for Efficient Finetuning » au lieu de « … of Large Language Models »). |
| Q4 | `rag_search` + `tavily_web_search` | `tavily_web_search`, 1 étape, 29.95 s | ≈ Partiel | Multi-hop non fait : `rag_search` ignoré malgré la règle du prompt ; réponse 100 % web, donc comparaison sans le corpus local. |
| Q5 | refus, 0 outil | refus poli, 0 outil, 8.24 s | ✅ Exact | Le garde-fou du system prompt fonctionne, à coût nul en outils. Refus très bref (pas d'explication du hors-domaine). |

### Enseignements

- **Q1 — le corpus local suffit.** L'agent route correctement la question la plus simple et reste fidèle à l'observation. C'est le cas idéal de ReAct : un seul outil, une seule étape.
- **Q2 — bon outil ≠ réponse parfaite.** La recherche web est bien choisie et les sources sont citées, mais la qualification « open-source » est approximative (licence non commerciale pour un des modèles).
- **Q3 — l'auto-correction est accidentelle.** La requête arXiv est trop vague, l'observation est hors sujet, et c'est un second outil (`rag_search`) qui sauve la réponse. Le titre du papier reste faux : l'agent n'a pas vérifié l'observation arXiv, il a répondu depuis le corpus local.
- **Q4 — les règles du prompt restent des suggestions.** Le prompt demande `rag_search` d'abord pour HNSW ; l'agent va directement sur le web. Aucun mécanisme ne l'oblige à faire le multi-hop attendu.
- **Q5 — le refus hors-domaine est fiable.** Une règle simple et non ambiguë est suivie, sans appel d'outil.
- **Contrainte opérationnelle.** Le tier gratuit de `gpt-oss-20b` plafonne à **8 000 TPM** : une exécution sans pause peut renvoyer `413 rate_limit_exceeded`. D'où `reasoning_effort="low"`, `max_tokens=1024` et la pause de 60 s avec nouvel essai dans la boucle.

### Conclusion (Partie 1)

Un agent ReAct pur n'a **aucun mécanisme d'auto-évaluation** : il ne vérifie ni la pertinence des documents récupérés, ni la fidélité de sa réponse, et ne change pas de source de lui-même en cas d'échec. Les résultats ci-dessus le montrent (Q3, Q4). La Partie 3 (LangGraph) corrige cela avec des nœuds `grade_docs` / `grade_answer`, une réécriture de requête et un fallback web imposés par le développeur.


In [48]:
# --- Migration LangChain 1.0 (récapitulatif des cellules 3 et 5) ---
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# Web tool (replaces TavilySearchResults)
tavily_tool = TavilySearch(
    max_results=3,
    name="tavily_web_search",
    description="Recherche sur le web public pour les informations récentes ou hors corpus.",
)

# La liste d'outils doit être reconstruite avec le nouveau tavily_tool
tools = [rag_search, tavily_tool, arxiv_search]

# Agent (replaces langgraph.prebuilt.create_react_agent)
agent_executor = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)
print(f"Agent ReAct (LangChain 1.x) reconstruit avec {len(tools)} outils.")


Agent ReAct (LangChain 1.x) reconstruit avec 3 outils.


In [49]:
# --- Exercice 3 : mesure automatique du routage ---
def routing_score(golden_set, react_results):
    points = {"exact": 1.0, "partiel": 0.5, "faux": 0.0}
    per_q, total = {}, 0.0
    for item in golden_set:
        exp = set(item["expected_tools"])
        got = set(react_results[item["id"]]["tools_used"])
        if exp == got:
            verdict = "exact"
        elif exp and got and (exp & got):
            verdict = "partiel"
        else:
            verdict = "faux"
        per_q[item["id"]] = verdict
        total += points[verdict]
    return per_q, round(total / len(golden_set), 2)


per_q, score = routing_score(golden_set, react_results)
print(per_q, "→ score global :", score)


{'Q1': 'exact', 'Q2': 'exact', 'Q3': 'partiel', 'Q4': 'partiel', 'Q5': 'exact'} → score global : 0.8
